#Dataset Dados Abertos STJ - Íntegras de Acórdãos do Diário da Justiça (Acórdãos e Decisões)

Pre-processing dos datasets por ano (único ou lista - 2021 ... 2026), parametrizado por tipo de decisão (monocrática ou acórdão) e por tipo de processo.

##Importação de bibliotecas e declaração de variáveis globais

In [17]:
ENVIRONMENT = 'COLAB' #'PUCPR'
TIPO_ARQUIVO = 'decisoes' #'acordaos'
ANO_CARGA = 2021
#'AREsp' #'REsp' #HC
CLASSE = 'HC'

#Não sendo informado o ANO
ANO_INI =2021
ANO_FIM= 2026

import torch

# Check if a GPU (CUDA) is available
if torch.cuda.is_available():
    print("GPU is available!")
    print("Device Name:", torch.cuda.get_device_name(0))
else:
    print("GPU not available, using CPU.")

# This is the line you'd use in your script to send your model or data to the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

GPU not available, using CPU.
Using device: cpu


##Parametrização conforme ambiente de execução.

In [18]:
from google.colab import drive
drive.mount('/content/drive')

%run "/content/drive/MyDrive/_Mestrado PUCPR/libs/config_environment.ipynb"

%run "/content/drive/MyDrive/_Mestrado PUCPR/libs/preprocessing_pucpr.ipynb"

Mounted at /content/drive
version 1.05.0


In [19]:
from datetime import datetime, date

def cast_string_to_date(ano: str, date_string: str) -> date:
    """
    Casts a string in 'YYYY-MM-DD' format to a datetime.date object.

    Args:
        date_string (str): The date string to cast, e.g., '2023-10-26'.

    Returns:
        datetime.date: The converted date object.

    Raises:
        ValueError: If the date_string is not in 'YYYY-MM-DD' format or is not a valid date.
    """
    try:
        if (ano == 2022):
          return datetime.strptime(date_string[0:10], '%d/%m/%Y').date()
        if (ano == 2023):
          return datetime.strptime(date_string, '%d/%m/%Y %H:%M:%S').date()
        else:
          return datetime.strptime(date_string, '%Y-%m-%d').date()
    except ValueError:
        raise ValueError(f"Invalid date string format or value: '{date_string}'. Expected 'YYYY-MM-DD'.")
        #return None

# Example usage:
# try:
#     date_obj = cast_string_to_date('2023-10-26')
#     print(f"Converted date: {date_obj} (Type: {type(date_obj)})")
#     date_obj_invalid = cast_string_to_date('2023/10/26') # This will raise a ValueError
# except ValueError as e:
#     print(e)

##Carregar classe/rótulo para a decisão (Favorável ou Não ao Réu)

In [20]:
# Define file paths
file_path = base_path + 'descricao-decisao-julgamento.csv'

# Load each CSV file into a dataframe
try:
    dfTipoDecisao = pd.read_csv(file_path, sep=';')
    print(f"Successfully loaded {file_path} into dataframe 'dfTipoDecisao'")
    display(dfTipoDecisao.head())
except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
except Exception as e:
    print(f"An error occurred while loading {file_path}: {e}")

Successfully loaded /content/drive/MyDrive/_Mestrado PUCPR/Datasets/descricao-decisao-julgamento.csv into dataframe 'dfTipoDecisao'


,COD_TEOR_DECISAO,DESC_TEOR_DECISAO,DESC_RSULT_JGTO,RESULTADO_RESUMIDO,DECISAO_FAVORAVEL
0,1,Concedendo,"AGRAVO CONHECIDO EM PARTE E, NESSA PARTE, PROVIDO",PARCIALMENTE CONHECIDO E PROVIDO,S
1,1,Concedendo,Agravo regimental provido,PROVIDO,S
2,1,Concedendo,CONCEDENDO HABEAS CORPUS,CONCEDIDO,S
3,1,Concedendo,CONCEDENDO HABEAS DATA,CONCEDIDO,S
4,1,Concedendo,CONCEDENDO MANDADO DE SEGURANÇA,CONCEDIDA,S


##Carregar metadados dos acórdão ou decisões (Documento, Processo, Nro de Registro, Ministro, Teor).

In [21]:
from datetime import datetime, date
import numpy as np
import pandas as pd

#tipo_arquivo: { acordaos = decisões colegiadas | decisoes = decisões monográticas }


def load_metadados(ano: int, tipo_arquivo: str = 'acordaos', classe: str = ''):

  def cast_string_to_date(date_string: str) -> date:
      """
      Casts a string in 'YYYY-MM-DD' format to a datetime.date object.

      Args:
          date_string (str): The date string to cast, e.g., '2023-10-26'.

      Returns:
          datetime.date: The converted date object.

      Raises:
          ValueError: If the date_string is not in 'YYYY-MM-DD' format or is not a valid date.
      """
      # Handle non-string values (like NaN floats) in date columns
      if not isinstance(date_string, str) and pd.isna(date_string):
          return pd.NaT # Return Not a Time for NaNs

      try:
          if (ano == 2022):
            return datetime.strptime(date_string[0:10], '%d/%m/%Y').date()
          if (ano == 2023):
            return datetime.strptime(date_string, '%d/%m/%Y %H:%M:%S').date()
          else:
            return datetime.strptime(date_string, '%Y-%m-%d').date()
      except ValueError:
          # If a ValueError still occurs for a string, return NaT as well
          return pd.NaT

  # Example usage:
  # try:
  #     date_obj = cast_string_to_date('2023-10-26')
  #     print(f"Converted date: {date_obj} (Type: {type(date_obj)})")
  #     date_obj_invalid = cast_string_to_date('2023/10/26') # This will raise a ValueError
  # except ValueError as e:
  #     print(e)

  def remove_blank(field):
    return field.strip() if isinstance(field, str) else field

  file_path = base_path + f'{ano}/metadados-{tipo_arquivo}-dj{ano}.csv'

  try:
      df_metadados = pd.read_csv(file_path, sep=';')
      print(f"Successfully loaded {file_path} into dataframe 'dfMetaDados-{tipo_arquivo}-' with {df_metadados.shape[0]} raws.")

      df_metadados['PROCESSO'].apply(remove_blank)
      df_metadados = df_metadados[df_metadados['PROCESSO'].astype(str).str.startswith(classe)]

      print(f"After clean for {classe} dataframe now has {df_metadados.shape[0]} raws.")

      #remover espaços em branco dos campos textuais
      df_metadados['TIPODOCUMENTO'].apply(remove_blank)
      df_metadados['NUMEROREGISTRO'].apply(remove_blank)

      df_metadados['RECURSO'].apply(remove_blank)
      df_metadados['TEOR'].apply(remove_blank)
      df_metadados['DESCRICAOMONOCRATICA'].apply(remove_blank)
      df_metadados['ASSUNTOS'].apply(remove_blank)

      df_metadados['DATAPUBLICACAO']   = df_metadados['DATAPUBLICACAO'].apply(cast_string_to_date)
      df_metadados['DATARECEBIMENTO']  = df_metadados['DATARECEBIMENTO'].apply(cast_string_to_date)
      df_metadados['DATADISTRIBUICAO'] = df_metadados['DATADISTRIBUICAO'].apply(cast_string_to_date)

      df_metadados['DATAPUBLICACAO'] = pd.to_datetime(df_metadados['DATAPUBLICACAO'])
      df_metadados['DATARECEBIMENTO'] = pd.to_datetime(df_metadados['DATARECEBIMENTO'])

      # Calculate the difference in days and convert to integer (allowing for NaNs)
      df_metadados['NUM_DIAS_CONCLUSAO_PROCESSO'] = (df_metadados['DATAPUBLICACAO'] - df_metadados['DATARECEBIMENTO']).dt.days.astype(pd.Int64Dtype())

      # Separar o número do processo pelo tipo
      df_metadados['TIPOPROCESSO'] = df_metadados['PROCESSO'].apply(lambda tipoprocesso: tipoprocesso.split(" ")[0])
      df_metadados['NUMEROPROCESSO'] = df_metadados['PROCESSO'].apply(lambda numeroprocesso: numeroprocesso.split(" ")[1])

      df_metadados.drop(columns=['PROCESSO'], inplace=True, axis=1)

      if (ano == 2024) and ('NM_MINISTRO' in df_metadados.columns) and (tipo_arquivo == 'acordaos'):
        print(" >> ajuste específico para 2024 e acordãos..")
        df_metadados.drop(columns=['MINISTRO'], inplace=True, axis=1)
        df_metadados.rename(columns={'NM_MINISTRO': 'MINISTRO'}, inplace=True)

      if (ano == 2021 or ano == 2025 or ano == 2026) and ('NM_MINISTRO' in df_metadados.columns):
        df_metadados.rename(columns={'NM_MINISTRO': 'MINISTRO'}, inplace=True)

      df_metadados['MINISTRO'].apply(remove_blank)

      #print("passou... ")
      #print(f"Total de linhas: {df_metadados.shape[0]}")

      return df_metadados
      #display(dfMetaDadosDecisao.head(3))
  except FileNotFoundError:
      print(f"Error: File not found at {file_path}")
      return pd.DataFrame() # Return an empty DataFrame on FileNotFoundError
  except Exception as e:
      print(f"An error occurred while loading {file_path}: {e}")
      return pd.DataFrame() # Return an empty DataFrame on any other Exception

In [22]:
#Teste isoaldo para observar resultados
# ano = 2026
# df2 = load_metadados(ano, 'acordaos', 'HC')
# df2.head()
#dfMetaDadosAcordao.head()

#dfDetalhesAcordao = load_dados_acordaos(ano)
#dfDetalhesAcordao.head()

# df2.head(3)

In [23]:
#Teste chamada direta
# ano = 2026
# file_path = base_path + f'{ano}/metadados-acordaos-dj{ano}.csv'
# print(file_path)

# df = pd.read_csv(file_path, sep=';')
# print(f"Successfully loaded {file_path} into dataframe 'dfMetaDadosDecisao' with {df.shape[0]} raws.")

In [24]:
# df.head()

In [25]:
#df[df['NM_MINISTRO'].notna()]

In [50]:
#Teste chamada direta dados textuais
ano = 2021
df_dados_teste = load_dados(ano, 'decisoes')
df_dados_teste.head()

/content/drive/MyDrive/_Mestrado PUCPR/Datasets/2021/dados-decisoes-dj2021.csv
Loading dataset (optimized for testing)...
Successfully loaded the first 50 rows of /content/drive/MyDrive/_Mestrado PUCPR/Datasets/2021/dados-decisoes-dj2021.csv


,SEQDOCUMENTO,ACORDAO,EMENTA,RELATORIO,VOTO,DECISAO
0,119739763,NaN,NaN,NaN,NaN,Cuida-se de habeas corpus com pedido de limina...
1,119752516,NaN,NaN,NaN,NaN,Cuida-se de habeas corpus com pedido de limina...
2,119755463,NaN,NaN,NaN,NaN,Cuida-se de recurso ordinário em habeas corpus...
3,119749665,NaN,NaN,NaN,NaN,Cuida-se de habeas corpus com pedido de limina...
4,119755373,NaN,NaN,NaN,NaN,Cuida-se de habeas corpus com pedido de limina...


In [71]:
def extrair_relatorio(x):
    if not isinstance(x, str):
        return x

    x_upper = x.upper()
    # O marcador agora obrigatoriamente inclui o ponto final
    marker_rel = 'O RELATÓRIO.'

    if marker_rel in x_upper:
        rel_pos = x_upper.find(marker_rel)

        # Retorna todo o conteúdo antes da frase
        return x[:rel_pos].strip()
    return x

def extrair_decisao(x):
    if not isinstance(x, str):
        return x

    x_upper = x.upper()
    # O marcador agora obrigatoriamente inclui o ponto final
    marker_rel = 'O RELATÓRIO.'

    if marker_rel in x_upper:
        rel_pos = x_upper.find(marker_rel)
        start_idx = rel_pos + len(marker_rel)

        # Retorna todo o conteúdo que segue após o ponto-final da frase
        return x[start_idx:].strip()
    return x

In [72]:
texto = """
Cuida-se de habeas corpus com pedido de liminar impetrado em favor de ALEXSANDRO DOS SANTOS SANTANAem que se aponta como autoridade coatora o TRIBUNAL DE JUSTIÇA DO ESTADO DA BAHIA(HCn. 8036997-30.2020.8.05.0000).<br>Opaciente foi preso preventivamente, em 16/12/2020, e denunciado pela prática das condutas descritas nos art. 1, inciso I, "a", da Lei n. 9.455/97, c/c art. 129, § 9º, c/c art. 71, caput, art. 147, caput, c/c art. 71, caput, e art. 163, parágrafo único, I, c/c art. 69 do CP, com incidência da Lei n. 11.340/06.<br>Oimpetrante sustentaque o paciente está sofrendo constrangimento ilegal em razão daausência de fundamentação concreta do decreto prisional, ausência de audiência de custódia, evidente excesso de prazo para apreciação de pedido de revogação de prisão preventiva, bem comolesão ao princípio da presunção de inocência.<br>Requer, liminarmente, a revogação da prisão preventiva; no mérito, pugna pela concessão da ordem para que seja confirmada a liminar.<br>É, no essencial, o relatório.
"""

decisao = extrair_decisao(texto)
print("decisão:" + decisao)

relatorio_content = extrair_relatorio(texto)
print("relatorio:" + relatorio_content)


decisão:
relatorio:Cuida-se de habeas corpus com pedido de liminar impetrado em favor de ALEXSANDRO DOS SANTOS SANTANAem que se aponta como autoridade coatora o TRIBUNAL DE JUSTIÇA DO ESTADO DA BAHIA(HCn. 8036997-30.2020.8.05.0000).<br>Opaciente foi preso preventivamente, em 16/12/2020, e denunciado pela prática das condutas descritas nos art. 1, inciso I, "a", da Lei n. 9.455/97, c/c art. 129, § 9º, c/c art. 71, caput, art. 147, caput, c/c art. 71, caput, e art. 163, parágrafo único, I, c/c art. 69 do CP, com incidência da Lei n. 11.340/06.<br>Oimpetrante sustentaque o paciente está sofrendo constrangimento ilegal em razão daausência de fundamentação concreta do decreto prisional, ausência de audiência de custódia, evidente excesso de prazo para apreciação de pedido de revogação de prisão preventiva, bem comolesão ao princípio da presunção de inocência.<br>Requer, liminarmente, a revogação da prisão preventiva; no mérito, pugna pela concessão da ordem para que seja confirmada a limina

##Carregar detalhes das decisões

In [49]:
def load_dados(ano: int, tipo_arquivo: str = 'acordaos'):
  # Define file paths
  file_path = base_path + f'{ano}/dados-{tipo_arquivo}-dj{ano}.csv'
  print(file_path)
  try:
      print("Loading dataset (optimized for testing)...")
      # Using nrows=50 to load only the necessary rows from disk
      df_dados_acordaos = pd.read_csv(file_path, sep=';', nrows=50)
      print(f"Successfully loaded the first 50 rows of {file_path}")

      # if (tipo_arquivo == 'acordaos'):
      #   print("ementa pre processing...")
      #   df_dados_acordaos['EMENTA'] = df_dados_acordaos['EMENTA'].apply(pre_processamento)

      #   print("relatorio pre processing...")
      #   df_dados_acordaos['RELATORIO'] = df_dados_acordaos['RELATORIO'].apply(pre_processamento)
      # elif tipo_arquivo == 'decisoes':
      #   print("Extraindo RELATORIO e DECISAO...")
      #   df_dados_acordaos['RELATORIO'] = df_dados_acordaos['DECISAO'].apply(extrair_relatorio)
      #   df_dados_acordaos['DECISAO'] = df_dados_acordaos['DECISAO'].apply(extrair_decisao)

      #   print("DECISAO e RELATORIO pre processing...")
      #   df_dados_acordaos['RELATORIO'] = df_dados_acordaos['RELATORIO'].apply(pre_processamento)
      #   df_dados_acordaos['DECISAO'] = df_dados_acordaos['DECISAO'].apply(pre_processamento)
      # else:
      #   raise Exception("Tipo de arquivo não suportado")

      return df_dados_acordaos
  except FileNotFoundError:
      print(f"Error: File not found at {file_path}")
  except Exception as e:
      print(f"An error occurred while loading {file_path}: {e}")

In [29]:
# df = load_dados(2021, TIPO_ARQUIVO)
# df

In [30]:
def join_datasets(dfMetaDados, dfDetalhes, tipo_arquivo):

  def get_decisao(seq_documento):
      row = dfDetalhes[dfDetalhes['SEQDOCUMENTO'] == seq_documento].head(1)
      if not row.empty:
          return row.head(1)['DECISAO'].iloc[0]
      else:
          return None

  def get_relatorio(seq_documento):
      row = dfDetalhes[dfDetalhes['SEQDOCUMENTO'] == seq_documento].head(1)
      if not row.empty:
          return row.head(1)['RELATORIO'].iloc[0]
      else:
          return None

  def get_acordao(seq_documento):
      row = dfDetalhes[dfDetalhes['SEQDOCUMENTO'] == seq_documento].head(1)
      if not row.empty:
          return row.head(1)['ACORDAO'].iloc[0]
      else:
          return None

  def get_ementa(seq_documento):
      row = dfDetalhes[dfDetalhes['SEQDOCUMENTO'] == seq_documento].head(1)
      if not row.empty:
          return row.head(1)['EMENTA'].iloc[0]
      else:
          return None

  def get_voto(seq_documento):
      field = dfDetalhes[dfDetalhes['SEQDOCUMENTO'] == seq_documento].head(1)
      if not field.empty:
          return field.head(1)['VOTO'].iloc[0]
      else:
          return None

  def get_tipo_decisao(teor_decisao):
      tipo_decisao_row = dfTipoDecisao[dfTipoDecisao['DESC_TEOR_DECISAO'] == teor_decisao].head(1)
      if not tipo_decisao_row.empty:
          return tipo_decisao_row.head(1)['DECISAO_FAVORAVEL'].iloc[0]
      else:
          return None

  dfMetaDados['DECISAO_FAVORAVEL']  = dfMetaDados['TEOR'].apply(get_tipo_decisao)

  if (tipo_arquivo == 'acordaos'):
    dfMetaDados['ACORDAO'] = dfMetaDados['SEQDOCUMENTO'].apply(get_acordao)
    dfMetaDados['EMENTA']  = dfMetaDados['SEQDOCUMENTO'].apply(get_ementa)
    dfMetaDados['RELATORIO']  = dfMetaDados['SEQDOCUMENTO'].apply(get_relatorio)
    dfMetaDados['VOTO']  = dfMetaDados['SEQDOCUMENTO'].apply(get_voto)
  elif (tipo_arquivo == 'decisoes'):
    dfMetaDados['DECISAO']  = dfMetaDados['SEQDOCUMENTO'].apply(get_decisao)
    dfMetaDados['RELATORIO']  = dfMetaDados['SEQDOCUMENTO'].apply(get_relatorio)
  else:
    raise Exception("Tipo de arquivo não suportado")

  return dfMetaDados

In [31]:
import gc

def clear_object(obj):
  # Delete the individual dataframes to free up memory
  del obj

  # Explicitly call the garbage collector
  gc.collect()

  print("Memory for df_metadados and df_dados has been released.")

In [32]:
# df_metadados.head()

In [33]:
all_df_joins = []

inicio, fim = 0, 0
if ANO_CARGA:
  inicio, fim = ANO_CARGA, ANO_CARGA+1 #apenas um ano
else:
  inicio, fim = ANO_INI, ANO_FIM+1 #todo período

print (f"Realizando carga de {inicio} a {fim} ...")
for ano in range(inicio, fim):
  print(f'Processing {ano} year...')

  print(f'   >> Processing MetaData for {ano}, {TIPO_ARQUIVO}, {CLASSE}...')
  df_metadados = load_metadados(ano, TIPO_ARQUIVO, CLASSE)
  #dfMetaDadosAcordao.head()

  print('   >> Processing Text Data...')
  df_dados = load_dados(ano, TIPO_ARQUIVO)
  #df_dados.head()

  print('   >> Join metadata and text data...')
  dfJoin = join_datasets(df_metadados, df_dados, TIPO_ARQUIVO)
  all_df_joins.append(dfJoin)

  clear_object(df_metadados)
  clear_object(df_dados)
  clear_object(dfJoin)

dataset = pd.concat(all_df_joins, ignore_index=True)
clear_object(all_df_joins)

dataset.head()

Realizando carga de 2021 a 2022 ...
Processing 2021 year...
   >> Processing MetaData for 2021, decisoes, HC...
Successfully loaded /content/drive/MyDrive/_Mestrado PUCPR/Datasets/2021/metadados-decisoes-dj2021.csv into dataframe 'dfMetaDados-decisoes-' with 432088 raws.
After clean for HC dataframe now has 81256 raws.
   >> Processing Text Data...
/content/drive/MyDrive/_Mestrado PUCPR/Datasets/2021/dados-decisoes-dj2021.csv
Loading dataset (optimized for testing)...
Successfully loaded the first 50 rows of /content/drive/MyDrive/_Mestrado PUCPR/Datasets/2021/dados-decisoes-dj2021.csv
Extraindo RELATORIO e DECISAO...
DECISAO e RELATORIO pre processing...
   >> Join metadata and text data...
Memory for df_metadados and df_dados has been released.
Memory for df_metadados and df_dados has been released.
Memory for df_metadados and df_dados has been released.
Memory for df_metadados and df_dados has been released.


,SEQDOCUMENTO,DATAPUBLICACAO,TIPODOCUMENTO,NUMEROREGISTRO,DATARECEBIMENTO,DATADISTRIBUICAO,MINISTRO,RECURSO,TEOR,DESCRICAOMONOCRATICA,ASSUNTOS,NUM_DIAS_CONCLUSAO_PROCESSO,TIPOPROCESSO,NUMEROPROCESSO,DECISAO_FAVORAVEL,DECISAO,RELATORIO
0,119739763,2021-01-04,DECISAO,202003489900,2020-12-27,2020-12-27,HUMBERTO MARTINS,NaN,Negando,Indeferido liminarmente o habeas corpus de #{...,"00287.03385.05560., 00287.03400.03402.",8,HC,637258,N,Cuida-se habeas corpus pedido liminar impetrad...,Cuida-se habeas corpus pedido liminar impetrad...
1,119752516,2021-01-04,DECISAO,202003496724,2020-12-29,2020-12-29,HUMBERTO MARTINS,NaN,Negando,Indeferido liminarmente o habeas corpus de #{...,"01209.04355., 00287.03415.05566., 12467.12612.",6,HC,637787,N,Cuida-se habeas corpus pedido liminar impetrad...,Cuida-se habeas corpus pedido liminar impetrad...
2,119749665,2021-01-04,DECISAO,202003494324,2020-12-29,2020-12-29,HUMBERTO MARTINS,NaN,Negando,Indeferido liminarmente o habeas corpus de #{...,"00287.03603.03607.03608., 00287.03603.03607.05...",6,HC,637685,N,Cuida-se habeas corpus pedido liminar impetrad...,Cuida-se habeas corpus pedido liminar impetrad...
3,119755373,2021-01-04,DECISAO,202003499970,2020-12-30,2020-12-30,HUMBERTO MARTINS,NaN,Negando,Indeferido liminarmente o habeas corpus de #{...,"01209.04355., 00287.03603.03607.03608., 12467....",5,HC,637950,N,Cuida-se habeas corpus pedido liminar impetrad...,Cuida-se habeas corpus pedido liminar impetrad...
4,119751685,2021-01-04,DECISAO,202003491456,2020-12-28,2020-12-28,HUMBERTO MARTINS,NaN,Negando,Indeferido liminarmente o habeas corpus de #{...,"01209.07942., 01209.07942.07791.",7,HC,637535,N,Cuida-se habeas corpus pedido liminar impetrad...,Cuida-se habeas corpus pedido liminar impetrad...


In [34]:
#df_metadados = load_metadados(2024, TIPO_ARQUIVO)

In [35]:
#df_metadados.shape

####Verificando distribuição dos dados propriamente dita

In [36]:
total   = dataset.shape[0]
label_s = dataset[dataset['DECISAO_FAVORAVEL'] == 'S'].shape[0]
label_n = dataset[dataset['DECISAO_FAVORAVEL'] == 'N'].shape[0]

print(f'TOTAL  : {total}')
print(f'LABEL S: {label_s} - % {round(label_s/total*100, 2)}')
print(f'LABEL N: {label_n} - % {round(label_n/total*100, 2)}')

TOTAL  : 81256
LABEL S: 23613 - % 29.06
LABEL N: 57643 - % 70.94


In [37]:
descricao_periodo = 'geral' if (ANO_CARGA == None) else ANO_CARGA
descricao_periodo

2021

In [38]:
CLASSE

'HC'

In [39]:
descricao_periodo = 'geral' if (ANO_CARGA is None) else str(ANO_CARGA)

output_csv_path = f"{base_path}preprocessed_{TIPO_ARQUIVO}_{CLASSE}_{descricao_periodo}_.csv"
print(output_csv_path)

# Save the DataFrame to a CSV file
dataset.to_csv(output_csv_path, index=False, sep=';')

print(f"Preprocessed dataset saved to: {output_csv_path}")
print("First 5 rows of the saved DataFrame:")
display(dataset.head())

/content/drive/MyDrive/_Mestrado PUCPR/Datasets/preprocessed_decisoes_HC_2021_.csv
Preprocessed dataset saved to: /content/drive/MyDrive/_Mestrado PUCPR/Datasets/preprocessed_decisoes_HC_2021_.csv
First 5 rows of the saved DataFrame:


,SEQDOCUMENTO,DATAPUBLICACAO,TIPODOCUMENTO,NUMEROREGISTRO,DATARECEBIMENTO,DATADISTRIBUICAO,MINISTRO,RECURSO,TEOR,DESCRICAOMONOCRATICA,ASSUNTOS,NUM_DIAS_CONCLUSAO_PROCESSO,TIPOPROCESSO,NUMEROPROCESSO,DECISAO_FAVORAVEL,DECISAO,RELATORIO
0,119739763,2021-01-04,DECISAO,202003489900,2020-12-27,2020-12-27,HUMBERTO MARTINS,NaN,Negando,Indeferido liminarmente o habeas corpus de #{...,"00287.03385.05560., 00287.03400.03402.",8,HC,637258,N,Cuida-se habeas corpus pedido liminar impetrad...,Cuida-se habeas corpus pedido liminar impetrad...
1,119752516,2021-01-04,DECISAO,202003496724,2020-12-29,2020-12-29,HUMBERTO MARTINS,NaN,Negando,Indeferido liminarmente o habeas corpus de #{...,"01209.04355., 00287.03415.05566., 12467.12612.",6,HC,637787,N,Cuida-se habeas corpus pedido liminar impetrad...,Cuida-se habeas corpus pedido liminar impetrad...
2,119749665,2021-01-04,DECISAO,202003494324,2020-12-29,2020-12-29,HUMBERTO MARTINS,NaN,Negando,Indeferido liminarmente o habeas corpus de #{...,"00287.03603.03607.03608., 00287.03603.03607.05...",6,HC,637685,N,Cuida-se habeas corpus pedido liminar impetrad...,Cuida-se habeas corpus pedido liminar impetrad...
3,119755373,2021-01-04,DECISAO,202003499970,2020-12-30,2020-12-30,HUMBERTO MARTINS,NaN,Negando,Indeferido liminarmente o habeas corpus de #{...,"01209.04355., 00287.03603.03607.03608., 12467....",5,HC,637950,N,Cuida-se habeas corpus pedido liminar impetrad...,Cuida-se habeas corpus pedido liminar impetrad...
4,119751685,2021-01-04,DECISAO,202003491456,2020-12-28,2020-12-28,HUMBERTO MARTINS,NaN,Negando,Indeferido liminarmente o habeas corpus de #{...,"01209.07942., 01209.07942.07791.",7,HC,637535,N,Cuida-se habeas corpus pedido liminar impetrad...,Cuida-se habeas corpus pedido liminar impetrad...


In [40]:
import pandas as pd

# Define the path to the preprocessed CSV file
input_csv_path = output_csv_path

# Load the CSV file into a DataFrame
try:
    df_loaded = pd.read_csv(input_csv_path, sep=';')
    print(f"Successfully loaded {input_csv_path} into dataframe 'df_loaded'")
    display(df_loaded.head())
except FileNotFoundError:
    print(f"Error: File not found at {input_csv_path}")
except Exception as e:
    print(f"An error occurred while loading {input_csv_path}: {e}")

Successfully loaded /content/drive/MyDrive/_Mestrado PUCPR/Datasets/preprocessed_decisoes_HC_2021_.csv into dataframe 'df_loaded'


/tmp/ipykernel_21597/1520829909.py:8: DtypeWarning: Columns (15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df_loaded = pd.read_csv(input_csv_path, sep=';')


,SEQDOCUMENTO,DATAPUBLICACAO,TIPODOCUMENTO,NUMEROREGISTRO,DATARECEBIMENTO,DATADISTRIBUICAO,MINISTRO,RECURSO,TEOR,DESCRICAOMONOCRATICA,ASSUNTOS,NUM_DIAS_CONCLUSAO_PROCESSO,TIPOPROCESSO,NUMEROPROCESSO,DECISAO_FAVORAVEL,DECISAO,RELATORIO
0,119739763,2021-01-04,DECISAO,202003489900,2020-12-27,2020-12-27,HUMBERTO MARTINS,NaN,Negando,Indeferido liminarmente o habeas corpus de #{...,"00287.03385.05560., 00287.03400.03402.",8,HC,637258,N,Cuida-se habeas corpus pedido liminar impetrad...,Cuida-se habeas corpus pedido liminar impetrad...
1,119752516,2021-01-04,DECISAO,202003496724,2020-12-29,2020-12-29,HUMBERTO MARTINS,NaN,Negando,Indeferido liminarmente o habeas corpus de #{...,"01209.04355., 00287.03415.05566., 12467.12612.",6,HC,637787,N,Cuida-se habeas corpus pedido liminar impetrad...,Cuida-se habeas corpus pedido liminar impetrad...
2,119749665,2021-01-04,DECISAO,202003494324,2020-12-29,2020-12-29,HUMBERTO MARTINS,NaN,Negando,Indeferido liminarmente o habeas corpus de #{...,"00287.03603.03607.03608., 00287.03603.03607.05...",6,HC,637685,N,Cuida-se habeas corpus pedido liminar impetrad...,Cuida-se habeas corpus pedido liminar impetrad...
3,119755373,2021-01-04,DECISAO,202003499970,2020-12-30,2020-12-30,HUMBERTO MARTINS,NaN,Negando,Indeferido liminarmente o habeas corpus de #{...,"01209.04355., 00287.03603.03607.03608., 12467....",5,HC,637950,N,Cuida-se habeas corpus pedido liminar impetrad...,Cuida-se habeas corpus pedido liminar impetrad...
4,119751685,2021-01-04,DECISAO,202003491456,2020-12-28,2020-12-28,HUMBERTO MARTINS,NaN,Negando,Indeferido liminarmente o habeas corpus de #{...,"01209.07942., 01209.07942.07791.",7,HC,637535,N,Cuida-se habeas corpus pedido liminar impetrad...,Cuida-se habeas corpus pedido liminar impetrad...


In [41]:
df_loaded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81256 entries, 0 to 81255
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   SEQDOCUMENTO                 81256 non-null  int64 
 1   DATAPUBLICACAO               81256 non-null  object
 2   TIPODOCUMENTO                81256 non-null  object
 3   NUMEROREGISTRO               81256 non-null  int64 
 4   DATARECEBIMENTO              81256 non-null  object
 5   DATADISTRIBUICAO             81253 non-null  object
 6   MINISTRO                     81256 non-null  object
 7   RECURSO                      4807 non-null   object
 8   TEOR                         81256 non-null  object
 9   DESCRICAOMONOCRATICA         81256 non-null  object
 10  ASSUNTOS                     81256 non-null  object
 11  NUM_DIAS_CONCLUSAO_PROCESSO  81256 non-null  int64 
 12  TIPOPROCESSO                 81256 non-null  object
 13  NUMEROPROCESSO               81

In [42]:
df_loaded.shape

(81256, 17)

In [43]:
!python --version

Python 3.12.13


In [44]:
import pandas as pd
import numpy as np
import torch

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"PyTorch version: {torch.__version__}")

Pandas version: 2.2.2
NumPy version: 2.0.2
PyTorch version: 2.11.0+cpu


In [45]:
#Exibir o conteúdo do RELATÓRIO a partir do campo NUMEROREGISTRO
#202002861050

In [46]:
record_to_display = 202503493057 # @param {type:"integer"}

filtered_record = dataset[dataset['NUMEROREGISTRO'] == record_to_display]

if not filtered_record.empty:
    relatorio_content = filtered_record['RELATORIO'].iloc[0]
    print(f"RELATORIO content for NUMEROREGISTRO {record_to_display}:\n")
    print(relatorio_content)
else:
    print(f"No record found with NUMEROREGISTRO: {record_to_display}")

No record found with NUMEROREGISTRO: 202503493057


In [47]:
dataset_with_relatorio = dataset[dataset['RELATORIO'].notna()]
display(dataset_with_relatorio.head())

,SEQDOCUMENTO,DATAPUBLICACAO,TIPODOCUMENTO,NUMEROREGISTRO,DATARECEBIMENTO,DATADISTRIBUICAO,MINISTRO,RECURSO,TEOR,DESCRICAOMONOCRATICA,ASSUNTOS,NUM_DIAS_CONCLUSAO_PROCESSO,TIPOPROCESSO,NUMEROPROCESSO,DECISAO_FAVORAVEL,DECISAO,RELATORIO
0,119739763,2021-01-04,DECISAO,202003489900,2020-12-27,2020-12-27,HUMBERTO MARTINS,NaN,Negando,Indeferido liminarmente o habeas corpus de #{...,"00287.03385.05560., 00287.03400.03402.",8,HC,637258,N,Cuida-se habeas corpus pedido liminar impetrad...,Cuida-se habeas corpus pedido liminar impetrad...
1,119752516,2021-01-04,DECISAO,202003496724,2020-12-29,2020-12-29,HUMBERTO MARTINS,NaN,Negando,Indeferido liminarmente o habeas corpus de #{...,"01209.04355., 00287.03415.05566., 12467.12612.",6,HC,637787,N,Cuida-se habeas corpus pedido liminar impetrad...,Cuida-se habeas corpus pedido liminar impetrad...
2,119749665,2021-01-04,DECISAO,202003494324,2020-12-29,2020-12-29,HUMBERTO MARTINS,NaN,Negando,Indeferido liminarmente o habeas corpus de #{...,"00287.03603.03607.03608., 00287.03603.03607.05...",6,HC,637685,N,Cuida-se habeas corpus pedido liminar impetrad...,Cuida-se habeas corpus pedido liminar impetrad...
3,119755373,2021-01-04,DECISAO,202003499970,2020-12-30,2020-12-30,HUMBERTO MARTINS,NaN,Negando,Indeferido liminarmente o habeas corpus de #{...,"01209.04355., 00287.03603.03607.03608., 12467....",5,HC,637950,N,Cuida-se habeas corpus pedido liminar impetrad...,Cuida-se habeas corpus pedido liminar impetrad...
4,119751685,2021-01-04,DECISAO,202003491456,2020-12-28,2020-12-28,HUMBERTO MARTINS,NaN,Negando,Indeferido liminarmente o habeas corpus de #{...,"01209.07942., 01209.07942.07791.",7,HC,637535,N,Cuida-se habeas corpus pedido liminar impetrad...,Cuida-se habeas corpus pedido liminar impetrad...


### Resumo da Etapa de Preparação do Dataset

Este documento apresenta um script desenvolvido na linguagem Python (versão 3.12), cuja finalidade é concluir a elaboração do *dataset* central desta pesquisa de mestrado. A principal tarefa aqui executada consiste na consolidação e integração de um vasto volume de dados jurídicos (metadados e teores de decisões), os quais foram extraídos na fase anterior do projeto por meio de técnicas de *web scraping*. Através do cruzamento e unificação de bases distintas (sub-datasets), o código junta os dados brutos em um único arquivo, estruturando-os em um formato relacional consistente para formar o *dataset* principal da pesquisa. Para viabilizar esse processo de integração de dados, fez-se o uso intensivo da biblioteca Pandas (versão 2.2.2) na manipulação dos *dataframes*, com o apoio complementar do NumPy (versão 2.0.2) e PyTorch (versão 2.10.0).

In [48]:
total_relatorio_filled = dataset['RELATORIO'].count()
total_relatorio_not_filled = dataset['RELATORIO'].isnull().sum()

print(f"Total records with 'RELATORIO' filled: {total_relatorio_filled}")
print(f"Total records with 'RELATORIO' not filled: {total_relatorio_not_filled}")

Total records with 'RELATORIO' filled: 48
Total records with 'RELATORIO' not filled: 81208
